# 🔐 🔐 Senhas em Risco: Big Data e Segurança Digital

---

**Disciplina:** Tópicos de Big Data em Python  
**Instituição:** Unimetrocamp Wyden — Campinas/SP  
**Grupo:**
- Eduardo Gombrade — Análise e Desenvolvimento
- Leandro Schiavo — Documentação
- João Vendito — Dashboard e Visualizações

---

## 📓 Notebook 04 — Dashboard Interativo de Segurança de Senhas

**Responsável:** João Vendito  

**Objetivo deste notebook:**  
Construir um dashboard interativo completo utilizando Plotly,
reunindo as principais visualizações do projeto em painéis
navegáveis e visualmente impactantes.

**Visualizações construídas:**
1. Painel de KPIs gerais do projeto
2. Gráfico de barras interativo — Força das senhas
3. Gráfico de dispersão — Comprimento vs Força
4. Sunburst — Composição das senhas
5. Treemap — Top senhas RockYou
6. Gráfico de linha interativo — Timeline de vazamentos
7. Gráfico de bolhas — Maiores vazamentos da história
8. Dashboard final reunindo todos os painéis


---
## ⚙️ CÉLULA 1 — Instalação e Configuração


In [2]:
# ============================================================
# INSTALAÇÃO E IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

# Caso esteja rodando no Colab pela primeira vez:
# !pip install plotly kaleido --quiet

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import os
import warnings

warnings.filterwarnings('ignore')

# --- Tema global dos gráficos Plotly ---
TEMA = 'plotly_dark'
COR_FUNDO    = '#0f0f1a'
COR_PAINEL   = '#1a1a2e'
COR_TEXTO    = '#e0e0ff'
COR_GRADE    = '#2a2a4a'
PALETA_FORCA = {'Fraca': '#ff4444', 'Média': '#ffaa00', 'Forte': '#44ff88'}

pio.templates.default = TEMA

os.makedirs('outputs/graficos', exist_ok=True)

# --- Carregamento dos dados ---
df_passwords = pd.read_parquet('data/processed/passwords_clean.parquet')
df_rockyou   = pd.read_parquet('data/processed/rockyou_clean.parquet')
df_breaches  = pd.read_parquet('data/processed/breaches_clean.parquet')
df_nordpass  = pd.read_parquet('data/processed/nordpass_clean.parquet')

print('✅ Ambiente configurado e dados carregados!')
print(f'   passwords : {len(df_passwords):,} registros')
print(f'   rockyou   : {len(df_rockyou):,} registros')
print(f'   breaches  : {len(df_breaches):,} registros')
print(f'   nordpass  : {len(df_nordpass):,} registros')

✅ Ambiente configurado e dados carregados!
   passwords : 669,598 registros
   rockyou   : 154 registros
   breaches  : 293 registros
   nordpass  : 30 registros


---
## 📊 CÉLULA 2 — Painel de KPIs Gerais

Cartões com os números mais impactantes do projeto.


In [3]:
# ============================================================
# PAINEL DE KPIs — INDICADORES-CHAVE DO PROJETO
# ============================================================

total           = len(df_passwords)
pct_fracas      = (df_passwords['forca_label'] == 'Fraca').sum() / total * 100
pct_instantaneo = (df_passwords['tempo_quebra'] == 'Instantâneo').sum() / total * 100
comp_medio      = df_passwords['comprimento'].mean()
pct_sem_simbolo = (df_passwords['tem_simbolo'] == 0).sum() / total * 100
total_breach    = df_breaches['registros_afetados'].sum() / 1e9  # em bilhões

fig = go.Figure()

kpis = [
    {'titulo': 'Senhas Analisadas',      'valor': f'{total:,}',          'subtitulo': 'registros no dataset',     'cor': '#7b68ee'},
    {'titulo': 'Senhas Fracas',          'valor': f'{pct_fracas:.1f}%',  'subtitulo': 'do total classificado',    'cor': '#ff4444'},
    {'titulo': 'Quebra Instantânea',     'valor': f'{pct_instantaneo:.1f}%','subtitulo': 'quebradas em < 1 seg', 'cor': '#ff8800'},
    {'titulo': 'Comprimento Médio',      'valor': f'{comp_medio:.1f}',   'subtitulo': 'caracteres por senha',     'cor': '#ffcc00'},
    {'titulo': 'Sem Símbolo Especial',   'valor': f'{pct_sem_simbolo:.0f}%','subtitulo': 'das senhas analisadas','cor': '#00ccff'},
    {'titulo': 'Registros Vazados',      'valor': f'{total_breach:.1f}B','subtitulo': 'em grandes incidentes',    'cor': '#ff4488'},
]

for i, kpi in enumerate(kpis):
    col = i % 3
    row = i // 3
    fig.add_annotation(
        x=col / 3 + 1/6,
        y=1 - row * 0.5 - 0.15,
        text=f"<b style='font-size:28px;color:{kpi['cor']}'>{kpi['valor']}</b><br>"
             f"<span style='font-size:13px;color:#aaaacc'>{kpi['titulo']}</span><br>"
             f"<span style='font-size:10px;color:#666688'>{kpi['subtitulo']}</span>",
        showarrow=False,
        xref='paper', yref='paper',
        align='center'
    )

fig.update_layout(
    title=dict(text='🔐 Dashboard — Segurança Digital de Senhas | KPIs Gerais',
               font=dict(size=18, color=COR_TEXTO), x=0.5),
    plot_bgcolor=COR_PAINEL,
    paper_bgcolor=COR_FUNDO,
    height=280,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    margin=dict(t=60, b=10, l=10, r=10)
)

fig.show()
print('✅ Painel de KPIs renderizado!')

✅ Painel de KPIs renderizado!


---
## 📊 CÉLULA 3 — Gráfico Interativo: Força das Senhas com Filtro


In [4]:
# ============================================================
# GRÁFICO INTERATIVO 1 — FORÇA DAS SENHAS
# ============================================================

# Agrupa por categoria de comprimento e força
df_agr = (
    df_passwords
    .groupby(['categoria_comprimento', 'forca_label'])
    .size()
    .reset_index(name='quantidade')
)

ordem_cat = ['Muito Curta (≤6)', 'Curta (7-8)', 'Média (9-12)', 'Longa (13-16)', 'Muito Longa (>16)']
df_agr['categoria_comprimento'] = pd.Categorical(df_agr['categoria_comprimento'], categories=ordem_cat, ordered=True)
df_agr = df_agr.sort_values('categoria_comprimento')

fig = px.bar(
    df_agr,
    x='categoria_comprimento',
    y='quantidade',
    color='forca_label',
    color_discrete_map=PALETA_FORCA,
    barmode='group',
    title='Distribuição de Força das Senhas por Categoria de Comprimento',
    labels={
        'categoria_comprimento': 'Categoria de Comprimento',
        'quantidade': 'Quantidade de Senhas',
        'forca_label': 'Força'
    },
    template=TEMA
)

fig.update_layout(
    paper_bgcolor=COR_FUNDO,
    plot_bgcolor=COR_PAINEL,
    height=480,
    legend=dict(title='Nível de Força', bgcolor=COR_PAINEL),
    xaxis=dict(gridcolor=COR_GRADE),
    yaxis=dict(gridcolor=COR_GRADE)
)

fig.update_traces(marker_line_width=0.5, marker_line_color='#0f0f1a')
fig.show()
print('✅ Gráfico interativo 1 renderizado!')

✅ Gráfico interativo 1 renderizado!


---
## 📊 CÉLULA 4 — Sunburst: Composição das Senhas

Visualização em formato de rosca hierárquica mostrando
como as senhas se dividem por força e tipos de caracteres.


In [5]:
# ============================================================
# SUNBURST — COMPOSIÇÃO HIERÁRQUICA DAS SENHAS
# ============================================================

# Cria rótulo de composição legível para cada senha
def rotulo_composicao(row):
    partes = []
    if row['tem_maiuscula']: partes.append('Maiúsc.')
    if row['tem_minuscula']: partes.append('Minúsc.')
    if row['tem_numero']:    partes.append('Números')
    if row['tem_simbolo']:   partes.append('Símbolos')
    return ' + '.join(partes) if partes else 'Sem tipo'

df_passwords['composicao'] = df_passwords.apply(rotulo_composicao, axis=1)

df_sun = (
    df_passwords
    .groupby(['forca_label', 'composicao'])
    .size()
    .reset_index(name='quantidade')
)

fig = px.sunburst(
    df_sun,
    path=['forca_label', 'composicao'],
    values='quantidade',
    color='forca_label',
    color_discrete_map=PALETA_FORCA,
    title='Composição das Senhas — Força × Tipos de Caracteres Utilizados',
    template=TEMA
)

fig.update_layout(
    paper_bgcolor=COR_FUNDO,
    height=580,
    title_x=0.5,
    title_font=dict(size=15, color=COR_TEXTO)
)

fig.update_traces(
    textfont_size=11,
    insidetextorientation='radial'
)

fig.show()
print('✅ Sunburst renderizado!')

✅ Sunburst renderizado!


---
## 📊 CÉLULA 5 — Treemap: Top Senhas RockYou


In [6]:
# ============================================================
# TREEMAP — TOP 30 SENHAS MAIS USADAS (ROCKYOU)
# ============================================================

top30 = df_rockyou.head(30).copy()
top30['label'] = top30['senha'] + '<br>' + top30['frequencia'].apply(lambda x: f'{x:,}x')

fig = px.treemap(
    top30,
    path=[px.Constant('RockYou 2009'), 'senha'],
    values='frequencia',
    color='frequencia',
    color_continuous_scale='YlOrRd',
    title='Top 30 Senhas Mais Usadas no Vazamento RockYou 2009',
    template=TEMA,
    hover_data={'pct_uso': ':.2f', 'comprimento': True}
)

fig.update_layout(
    paper_bgcolor=COR_FUNDO,
    height=550,
    title_x=0.5,
    title_font=dict(size=15, color=COR_TEXTO),
    coloraxis_colorbar=dict(
        title='Frequência',
        tickfont=dict(color=COR_TEXTO)
    )
)

fig.update_traces(
    textfont=dict(size=13),
    marker_line_width=1.5,
    marker_line_color=COR_FUNDO
)

fig.show()
print('✅ Treemap renderizado!')

✅ Treemap renderizado!


---
## 📊 CÉLULA 6 — Linha Interativa: Timeline de Vazamentos


In [7]:
# ============================================================
# LINHA INTERATIVA — TIMELINE DE VAZAMENTOS (2004–2024)
# ============================================================

col_ano       = 'ano'               if 'ano'               in df_breaches.columns else 'year'
col_registros = 'registros_afetados' if 'registros_afetados' in df_breaches.columns else 'records'
col_setor     = 'setor'             if 'setor'             in df_breaches.columns else 'organization_type'

por_ano = (
    df_breaches
    .groupby(col_ano)
    .agg(
        incidentes=(col_ano, 'count'),
        total_afetados=(col_registros, 'sum')
    )
    .reset_index()
)
por_ano.columns = ['ano', 'incidentes', 'total_afetados']
por_ano['afetados_M'] = por_ano['total_afetados'] / 1e6

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=('Número de Incidentes por Ano', 'Total de Registros Afetados (Milhões)'),
    vertical_spacing=0.12
)

# Painel 1 — Incidentes
fig.add_trace(
    go.Bar(
        x=por_ano['ano'].astype(str),
        y=por_ano['incidentes'],
        name='Incidentes',
        marker_color='#7b68ee',
        hovertemplate='Ano: %{x}<br>Incidentes: %{y}<extra></extra>'
    ),
    row=1, col=1
)

# Painel 2 — Registros afetados
fig.add_trace(
    go.Scatter(
        x=por_ano['ano'].astype(str),
        y=por_ano['afetados_M'],
        name='Registros Afetados (M)',
        mode='lines+markers',
        line=dict(color='#ff4466', width=2.5),
        marker=dict(size=8, color='#ff4466'),
        fill='tozeroy',
        fillcolor='rgba(255, 68, 102, 0.15)',
        hovertemplate='Ano: %{x}<br>Afetados: %{y:.1f}M<extra></extra>'
    ),
    row=2, col=1
)

fig.update_layout(
    title=dict(text='Timeline de Grandes Vazamentos de Dados — 2004 a 2024',
               font=dict(size=15, color=COR_TEXTO), x=0.5),
    paper_bgcolor=COR_FUNDO,
    plot_bgcolor=COR_PAINEL,
    height=600,
    showlegend=True,
    legend=dict(bgcolor=COR_PAINEL)
)

fig.update_xaxes(gridcolor=COR_GRADE)
fig.update_yaxes(gridcolor=COR_GRADE)

fig.show()
print('✅ Timeline interativa renderizada!')

✅ Timeline interativa renderizada!


---
## 📊 CÉLULA 7 — Gráfico de Bolhas: Maiores Vazamentos da História


In [8]:
# ============================================================
# GRÁFICO DE BOLHAS — MAIORES VAZAMENTOS DA HISTÓRIA
# ============================================================

col_empresa   = 'empresa'           if 'empresa'           in df_breaches.columns else 'entity'
col_ano       = 'ano'               if 'ano'               in df_breaches.columns else 'year'
col_registros = 'registros_afetados' if 'registros_afetados' in df_breaches.columns else 'records'
col_setor     = 'setor'             if 'setor'             in df_breaches.columns else 'organization_type'

# Top 20 maiores vazamentos
top_breaches = (
    df_breaches
    .nlargest(20, col_registros)
    .copy()
)
top_breaches['afetados_M'] = top_breaches[col_registros] / 1e6
top_breaches['ano_str']    = top_breaches[col_ano].astype(str)

fig = px.scatter(
    top_breaches,
    x=col_ano,
    y='afetados_M',
    size='afetados_M',
    color=col_setor,
    hover_name=col_empresa,
    size_max=80,
    title='Top 20 Maiores Vazamentos de Dados da História (por volume de registros)',
    labels={
        col_ano       : 'Ano',
        'afetados_M'  : 'Registros Afetados (Milhões)',
        col_setor     : 'Setor'
    },
    template=TEMA,
    color_discrete_sequence=px.colors.qualitative.Vivid
)

fig.update_layout(
    paper_bgcolor=COR_FUNDO,
    plot_bgcolor=COR_PAINEL,
    height=560,
    title_x=0.5,
    title_font=dict(size=14, color=COR_TEXTO),
    xaxis=dict(gridcolor=COR_GRADE, title='Ano do Vazamento'),
    yaxis=dict(gridcolor=COR_GRADE, title='Registros Afetados (M)'),
    legend=dict(bgcolor=COR_PAINEL, title='Setor')
)

fig.show()
print('✅ Gráfico de bolhas renderizado!')

✅ Gráfico de bolhas renderizado!


---
## 📊 CÉLULA 8 — Gráfico de Barras: Top Senhas NordPass por País


In [9]:
# ============================================================
# BARRAS ANIMADAS — TOP SENHAS NORDPASS POR PAÍS
# ============================================================

fig = px.bar(
    df_nordpass,
    x='usuarios_afetados',
    y='senha',
    color='pais_mais_comum',
    orientation='h',
    title='Top 30 Senhas Mais Comuns no Mundo — NordPass 2024',
    labels={
        'usuarios_afetados' : 'Usuários Afetados',
        'senha'             : 'Senha',
        'pais_mais_comum'   : 'País de Destaque'
    },
    template=TEMA,
    color_discrete_sequence=px.colors.qualitative.Pastel,
    text='tempo_para_quebrar'
)

fig.update_layout(
    paper_bgcolor=COR_FUNDO,
    plot_bgcolor=COR_PAINEL,
    height=700,
    title_x=0.5,
    title_font=dict(size=14, color=COR_TEXTO),
    yaxis=dict(autorange='reversed', gridcolor=COR_GRADE),
    xaxis=dict(gridcolor=COR_GRADE),
    legend=dict(bgcolor=COR_PAINEL)
)

fig.update_traces(
    textposition='outside',
    textfont=dict(size=9, color='#ffcc44'),
    marker_line_width=0.5,
    marker_line_color=COR_FUNDO
)

fig.show()
print('✅ Gráfico NordPass renderizado!')

✅ Gráfico NordPass renderizado!


---
## 📊 CÉLULA 9 — Dashboard Final Consolidado

Painel único reunindo 4 visualizações simultâneas.


In [12]:
# ============================================================
# DASHBOARD FINAL — 4 PAINÉIS SIMULTÂNEOS
# ============================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        '🔐 Força das Senhas',
        '⏱ Tempo de Quebra',
        '📅 Incidentes por Ano',
        '🏆 Top 10 RockYou'
    ),
    specs=[
        [{"type": "domain"}, {"type": "xy"}],
        [{"type": "xy"},     {"type": "xy"}]
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.10
)

# --- Painel 1: Força das senhas (pizza) ---
contagem_forca = df_passwords['forca_label'].value_counts().reindex(['Fraca', 'Média', 'Forte'])
fig.add_trace(
    go.Pie(
        labels=contagem_forca.index,
        values=contagem_forca.values,
        marker_colors=[PALETA_FORCA[k] for k in contagem_forca.index],
        textinfo='label+percent',
        hole=0.35,
        showlegend=False
    ),
    row=1, col=1
)

# --- Painel 2: Tempo de quebra (barras) ---
ordem_tempo = ['Instantâneo','Segundos','Minutos','Horas','Dias','Semanas','Meses','Anos','Décadas','Séculos']
cont_tempo = df_passwords['tempo_quebra'].value_counts()
cont_tempo = cont_tempo.reindex([t for t in ordem_tempo if t in cont_tempo.index]).dropna()
cores_grad = px.colors.sequential.Plasma[::-1][:len(cont_tempo)]

fig.add_trace(
    go.Bar(
        x=list(cont_tempo.index),
        y=list(cont_tempo.values),
        marker_color=cores_grad,
        showlegend=False,
        hovertemplate='%{x}: %{y:,}<extra></extra>'
    ),
    row=1, col=2
)

# --- Painel 3: Incidentes por ano (linha) ---
col_ano = 'ano' if 'ano' in df_breaches.columns else 'year'
por_ano_dash = df_breaches.groupby(col_ano).size().reset_index(name='incidentes')
por_ano_dash.columns = ['ano', 'incidentes']

fig.add_trace(
    go.Scatter(
        x=por_ano_dash['ano'].astype(str),
        y=por_ano_dash['incidentes'],
        mode='lines+markers',
        line=dict(color='#7b68ee', width=2),
        marker=dict(size=6),
        showlegend=False,
        hovertemplate='Ano %{x}: %{y} incidentes<extra></extra>'
    ),
    row=2, col=1
)

# --- Painel 4: Top 10 RockYou (barras horizontais) ---
top10 = df_rockyou.head(10)
fig.add_trace(
    go.Bar(
        x=top10['frequencia'][::-1],
        y=top10['senha'][::-1],
        orientation='h',
        marker_color=px.colors.sequential.YlOrRd[::-1][:10],
        showlegend=False,
        hovertemplate='%{y}: %{x:,} usos<extra></extra>'
    ),
    row=2, col=2
)

fig.update_layout(
    title=dict(
        text='🔐 DASHBOARD — Segurança Digital de Senhas | Visão Consolidada',
        font=dict(size=16, color=COR_TEXTO), x=0.5
    ),
    paper_bgcolor=COR_FUNDO,
    plot_bgcolor=COR_PAINEL,
    height=780,
    font=dict(color=COR_TEXTO)
)

fig.update_xaxes(gridcolor=COR_GRADE, tickangle=30)
fig.update_yaxes(gridcolor=COR_GRADE)

fig.show()
print('✅ Dashboard final consolidado renderizado!')
print('\n🏁 Notebook 04 concluído!')
print('   → Próximo passo: Notebook 05 — Modelo Preditivo de Força de Senhas')

✅ Dashboard final consolidado renderizado!

🏁 Notebook 04 concluído!
   → Próximo passo: Notebook 05 — Modelo Preditivo de Força de Senhas
